In [1]:
import numpy as np
import os
import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder



In [2]:
def load_features():
    """
    Load EEG feature data for prediction.features are stored in 'data/features'.
    """
    features_dir = 'data/features'
    X = []
    filenames = []

    for file in os.listdir(features_dir):
        if file.endswith('.npy'):
            features = np.load(os.path.join(features_dir, file))
            X.append(features)
            filenames.extend([file] * len(features))

    return np.vstack(X), filenames

In [3]:
def load_csv_features(csv_path):
    try:
        # Load CSV data
        df = pd.read_csv(csv_path)
        
        # Separate features and labels
        # last column is the label
        if 'label' in df.columns:
            y = df['label'].values
            X = df.drop('label', axis=1).values
        else:
            # If no label column, assume all columns are features
            X = df.values
            y = None
            
        print(f"Loaded {len(X)} samples with {X.shape[1]} features from CSV")
        return X, y
        
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return None, None

In [4]:
def main():
    # Load SVM model and preprocessing components
    model_path = 'models/svm_model.joblib'
    scaler_path = 'models/scaler.joblib'
    label_encoder_path = 'models/label_encoder.joblib'
    feature_columns_path = 'models/feature_columns.joblib'
    
    # Check if all required files exist
    required_files = [model_path, scaler_path, label_encoder_path, feature_columns_path]
    for file_path in required_files:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Required file not found: {file_path}. Run train_model.py first.")

    # Load all components
    svm_model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    label_encoder = joblib.load(label_encoder_path)
    feature_columns = joblib.load(feature_columns_path)
    
    print("Loaded SVM model and preprocessing components successfully.")
    print(f"Model classes: {label_encoder.classes_}")

    # Load CSV feature data
    csv_path = r"D:\6th sem internship\archive (3)\emotions.csv"  
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found at {csv_path}.")

    X, y = load_csv_features(csv_path)
    if X is None:
        print(" Failed to load CSV data")
        return
        
    print(f"Loaded {len(X)} samples from CSV for prediction.")

    #  Preprocess the data (same as training)
    # i need to ensure it has the same features as training
    if X.shape[1] != len(feature_columns):
        print(f"Warning: Expected {len(feature_columns)} features, got {X.shape[1]}")
       
    # Scale the features
    X_scaled = scaler.transform(X)
    print("Applied feature scaling.")

    # Perform predictions
    predictions_encoded = svm_model.predict(X_scaled)
    probabilities = svm_model.predict_proba(X_scaled)
    
    # Decode predictions back to original labels
    predictions = label_encoder.inverse_transform(predictions_encoded)

    # Display results
    print("\nPrediction Results:")
    for i in range(len(predictions)):
        probs = ", ".join([f"{cls}: {prob:.2f}" for cls, prob in zip(label_encoder.classes_, probabilities[i])])
        actual_label = y[i] if y is not None else "Unknown"
        print(f"Sample {i+1} -> Actual Label: {actual_label} -> Predicted Label: {predictions[i]} -> Probabilities: [{probs}]")

    # Save predictions to file
    output_file = 'models/csv_detection_results.txt'
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("EEG Emotion Detection Results\n")
        f.write("=" * 50 + "\n\n")
        for i in range(len(predictions)):
            probs = ", ".join([f"{cls}: {prob:.2f}" for cls, prob in zip(label_encoder.classes_, probabilities[i])])
            actual_label = y[i] if y is not None else "Unknown"
            f.write(f"Sample {i+1} -> Actual Label: {actual_label} -> Predicted Label: {predictions[i]} -> Probabilities: [{probs}]\n")

    print(f"\n Detection results saved to {output_file}")

if __name__ == "__main__":
    main()


Loaded SVM model and preprocessing components successfully.
Model classes: ['NEGATIVE' 'NEUTRAL' 'POSITIVE']
Loaded 2132 samples with 2548 features from CSV
Loaded 2132 samples from CSV for prediction.
Applied feature scaling.

Prediction Results:
Sample 1 -> Actual Label: NEGATIVE -> Predicted Label: NEGATIVE -> Probabilities: [NEGATIVE: 1.00, NEUTRAL: 0.00, POSITIVE: 0.00]
Sample 2 -> Actual Label: NEUTRAL -> Predicted Label: NEUTRAL -> Probabilities: [NEGATIVE: 0.00, NEUTRAL: 1.00, POSITIVE: 0.00]
Sample 3 -> Actual Label: POSITIVE -> Predicted Label: POSITIVE -> Probabilities: [NEGATIVE: 0.05, NEUTRAL: 0.00, POSITIVE: 0.95]
Sample 4 -> Actual Label: POSITIVE -> Predicted Label: POSITIVE -> Probabilities: [NEGATIVE: 0.01, NEUTRAL: 0.00, POSITIVE: 0.99]
Sample 5 -> Actual Label: NEUTRAL -> Predicted Label: NEUTRAL -> Probabilities: [NEGATIVE: 0.00, NEUTRAL: 0.99, POSITIVE: 0.01]
Sample 6 -> Actual Label: NEUTRAL -> Predicted Label: NEUTRAL -> Probabilities: [NEGATIVE: 0.00, NEUTRAL: 